In [1]:
import duckdb as db, pandas as pd 

In [6]:
def _load_daily(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df = df[pd.to_datetime(df["Usage date"], format="%m/%d/%Y", errors="coerce").notna()].copy()
    df[["Meter read date", "Usage date"]] = df[["Meter read date", "Usage date"]].apply(
        pd.to_datetime, format="%m/%d/%Y"
    )
    return df

df_dailyusage = _load_daily("data/dailyUsage4_18_2023_to_4_17_2026.csv")
df_dailyusage["Total kWh"] = pd.to_numeric(
    df_dailyusage["Total kWh"].astype(str).str.replace(",", "").str.strip('"'),
    errors="coerce",
)

df_dailycost = _load_daily("data/dailyCost4_18_2023_to_4_17_2026.csv")
df_dailycost["Total cost"] = pd.to_numeric(
    df_dailycost["Total cost"].astype(str).str.replace("$", "", regex=False).str.strip('"'),
    errors="coerce",
)

print(df_dailyusage.dtypes)
df_dailyusage.head()

Meter read date         datetime64[ns]
Usage date              datetime64[ns]
Total kWh                        int64
High temperature (F)           float64
Low temperature (F)            float64
dtype: object


,Meter read date,Usage date,Total kWh,High temperature (F),Low temperature (F)
0,2023-04-19,2023-04-18,17,85.0,62.0
1,2023-04-20,2023-04-19,12,79.0,63.0
2,2023-04-21,2023-04-20,12,85.0,56.0
3,2023-04-22,2023-04-21,15,89.0,61.0
4,2023-04-23,2023-04-22,21,92.0,62.0


In [7]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_dailycost, title="Daily Cost EDA Report", explorative=True)
profile.to_notebook_iframe()

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 59.33it/s]


Key takeaways: 
1. higher temps -> higher cost, lower temps -> lower cost.
2. .75 Correlation R  = R2 56% explainability for variation in cost by highest temperature of the day. Still leaves other 44%. 

In [9]:
profile = ProfileReport(df_dailyusage, title="Daily usage EDA Report", explorative=True)
profile.to_notebook_iframe()

Render HTML: 100%|██████████| 1/1 [00:00<00:00, 59.57it/s]
